In [6]:
import kagglehub  # pyright: ignore[reportMissingImports]
import os
import pandas as pd

path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
print("Path to dataset files:", path)
print(os.listdir(path))

Resuming download from 69206016 bytes (107566657 bytes left)...
Resuming download to C:\Users\ASUS\.cache\kagglehub\datasets\thoughtvector\customer-support-on-twitter\10.archive (69206016/176772673) bytes left.


100%|██████████| 169M/169M [04:39<00:00, 385kB/s]  

Extracting files...


Path to dataset files: C:\Users\ASUS\.cache\kagglehub\datasets\thoughtvector\customer-support-on-twitter\versions\10
['sample.csv', 'twcs']


In [3]:
%pip install pandas numpy scikit-learn sentence-transformers ollama kagglehub



   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   -------------------- ------------------- 1/2 [kagglehub]
   ---------------------------------------- 2/2 [kagglehub]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
twcs_folder = os.path.join(path, "twcs")
print(os.listdir(twcs_folder))

['twcs.csv']


In [8]:
csv_path = os.path.join(path, "twcs", "twcs.csv")
df = pd.read_csv(csv_path)
df.to_csv("data/twcs.csv", index=False)
print(df.shape)
df.head(5)

(2811774, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [9]:
brand_counts = df[df['inbound'] == False]['author_id'].value_counts()
print(brand_counts.head(20))

author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
Name: count, dtype: int64


In [10]:
BRAND = "sprintcare"   

brand_tweets = df[df['author_id'] == BRAND].copy()
df['tweet_id_str'] = df['tweet_id'].astype(str)
brand_tweets['in_response_to_tweet_id'] = brand_tweets['in_response_to_tweet_id'].astype(str)

pairs = brand_tweets.merge(
    df[['tweet_id_str', 'text']].rename(columns={'tweet_id_str': 'in_response_to_tweet_id', 'text': 'customer_text'}),
    on='in_response_to_tweet_id',
    how='inner'
)
pairs = pairs.rename(columns={'text': 'brand_reply'})[['tweet_id', 'customer_text', 'brand_reply', 'created_at']]
print(pairs.shape)
pairs.head(5)

(0, 4)


,tweet_id,customer_text,brand_reply,created_at


In [11]:
pairs_sample = pairs.sample(n=min(5000, len(pairs)), random_state=42).reset_index(drop=True)
pairs_sample.to_csv("data/brand_pairs_sample.csv", index=False)
print("saved", len(pairs_sample), "pairs")

saved 0 pairs


In [12]:
BRAND = "sprintcare"

brand_tweets = df[df['author_id'] == BRAND].copy()
df['tweet_id_str'] = df['tweet_id'].astype(str)

# THE FIX: convert through Int64 (nullable int) before turning into strings,
# so "4.0" doesn't happen -- we get clean "4" instead
brand_tweets['in_response_to_tweet_id'] = (
    pd.to_numeric(brand_tweets['in_response_to_tweet_id'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

pairs = brand_tweets.merge(
    df[['tweet_id_str', 'text']].rename(columns={'tweet_id_str': 'in_response_to_tweet_id', 'text': 'customer_text'}),
    on='in_response_to_tweet_id',
    how='inner'
)
pairs = pairs.rename(columns={'text': 'brand_reply'})[['tweet_id', 'customer_text', 'brand_reply', 'created_at']]
print(pairs.shape)
pairs.head(5)

(22313, 4)


,tweet_id,customer_text,brand_reply,created_at
0,1,@sprintcare I have sent several private messag...,@115712 I understand. I would like to assist y...,Tue Oct 31 22:10:47 +0000 2017
1,4,@sprintcare I did.,@115712 Please send us a Private Message so th...,Tue Oct 31 21:54:49 +0000 2017
2,6,@sprintcare is the worst customer service,@115712 Can you please send us a private messa...,Tue Oct 31 21:46:24 +0000 2017
3,11,@sprintcare You gonna magically change your co...,@115713 This is saddening to hear. Please shoo...,Tue Oct 31 22:10:35 +0000 2017
4,15,@sprintcare Since I signed up with you....Sinc...,@115713 We understand your concerns and we'd l...,Tue Oct 31 20:03:31 +0000 2017


In [13]:
pairs_sample = pairs.sample(n=min(5000, len(pairs)), random_state=42).reset_index(drop=True)
pairs_sample.to_csv("data/brand_pairs_sample.csv", index=False)
print("saved", len(pairs_sample), "pairs")

saved 5000 pairs


In [14]:
import json
scores = [json.loads(l) for l in open("eval/judge_scores.jsonl", encoding="utf-8")]
none_count = sum(1 for s in scores if s['groundedness'] is None)
print(f"{none_count} / {len(scores)} judge calls failed to parse")

109 / 200 judge calls failed to parse


In [15]:
import json
scores = [json.loads(l) for l in open("eval/judge_scores.jsonl", encoding="utf-8")]
golden = {g['id']: g for g in (json.loads(l) for l in open("eval/golden_set.jsonl", encoding="utf-8"))}
preds = {p['id']: p for p in (json.loads(l) for l in open("eval/system_predictions.jsonl", encoding="utf-8"))}

valid_scores = [s for s in scores if s['groundedness'] is not None][:15]
for s in valid_scores:
    g = golden[s['id']]
    p = preds[s['id']]
    print(f"\n--- id {s['id']} ---")
    print("Customer:", g['customer_text'])
    print("Generated reply:", p['pred_reply'])
    print("Judge scores:", s)


--- id 1 ---
Customer: @sprintcare my mom lives in Taylorsville, KY by herself. She cannot place or receive calls. What is going on??? This is unacceptable.
Generated reply: "@123456 Hello there! Please follow us at @sprintcare and send us a DM, so we can further assist you in getting your mom's service up and running as expected. -DG"
Judge scores: {'groundedness': 1, 'correctness': 2, 'actionability': 1, 'id': 1}

--- id 2 ---
Customer: @sprintcare Then do what you have to do, I'm so tired of dealing with it at this poibt, it's been every day for over a week.  My patience is thin.
Generated reply: "@123456 I completely understand your frustration, Sir. I'm here to help and would like to assist you in resolving the issue as soon as possible. Can you please send me a private message with a brief update on what's been happening?"
Judge scores: {'groundedness': 3, 'correctness': 4, 'actionability': 4, 'id': 2}

--- id 6 ---
Customer: @175773 @115714 It's spotty at best, most places. Eve